# 智慧垃圾分類 — YOLO11s 訓練
**執行前請先確認：Runtime → Change runtime type → T4 GPU**

In [ ]:
# ── Cell 1：確認 GPU ──────────────────────────────────────
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
else:
    print('❌ 沒有 GPU，請先到 Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Cell 2：安裝依賴 ──────────────────────────────────────
!pip install ultralytics kaggle -q

In [ ]:
# ── Cell 3：從 GitHub 取得資料準備腳本 ────────────────────
!git clone https://github.com/Saibusu/AI-course.git
%cd AI-course

## 下載 TrashNet 資料集

### 方法一：Kaggle API Token（推薦）
1. 到 https://www.kaggle.com/settings → API → 複製 API Token（格式：`KGAT_...`）
2. 將 Token 貼到 Cell 4A 的 `YOUR_KAGGLE_API_TOKEN` 位置後執行

### 方法二：手動上傳
1. 到 https://github.com/garythung/trashnet → Releases → 下載 `dataset-resized.7z`
2. 上傳到 Colab，執行 Cell 4B

In [ ]:
# ── Cell 4A：Kaggle API Token 下載 TrashNet ──────────────
import os

# 到 kaggle.com/settings → API → 複製你的 Token，貼在下方引號內
os.environ['KAGGLE_TOKEN'] = 'YOUR_KAGGLE_API_TOKEN'

!kaggle datasets download -d asdasdasasdas/garbage-classification -p data/
!unzip -q data/garbage-classification.zip -d data/TrashNet
!ls data/TrashNet/

In [ ]:
# ── Cell 4B：手動上傳 dataset-resized.7z（方法二）─────────
# 如果用方法一成功，跳過此 Cell
from google.colab import files
print('請上傳 dataset-resized.7z：')
files.upload()

!apt-get install -q p7zip-full
!7z x dataset-resized.7z -o data/TrashNet/ -y
!ls data/TrashNet/

In [ ]:
# ── Cell 5：TrashNet → YOLO 6-class 格式轉換 ─────────────
!python data/prepare_trashnet.py \
    --trashnet-dir data/TrashNet \
    --output-dir   data/trashnet_yolo

# 確認資料集結構
import os
for split in ['train', 'val', 'test']:
    imgs = len(list(os.scandir(f'data/trashnet_yolo/{split}/images')))
    print(f'{split}: {imgs} images')

In [ ]:
# ── Cell 6：（選用）同時下載 TACO 資料集並合併 ────────────
# TACO 需要從 Flickr 下載圖片，速度較慢。有時間再跑。
# 如果只用 TrashNet，直接跳到 Cell 7。

!git clone https://github.com/pedropro/TACO.git data/TACO_repo
!pip install requests Pillow -q
!python data/TACO_repo/downloader.py \
    --dataset_path data/TACO/data \
    --ann_file data/TACO_repo/data/annotations.json

!python data/prepare_taco.py \
    --taco-dir   data/TACO_repo \
    --output-dir data/taco_yolo

!python data/merge_datasets.py \
    --taco     data/taco_yolo \
    --trashnet data/trashnet_yolo \
    --output   data/merged

In [ ]:
# ── Cell 7：開始訓練 ──────────────────────────────────────
from ultralytics import YOLO

# 如果有跑 Cell 6（合併資料集），改用 data/merged/data.yaml
DATA_YAML = 'data/trashnet_yolo/data.yaml'

model = YOLO('yolo11s.pt')  # 自動下載預訓練權重

results = model.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=416,
    batch=16,
    device=0,
    project='runs/train',
    name='waste_sorter',
    exist_ok=True,
    patience=15,
    lr0=1e-3,
    lrf=1e-2,
    mosaic=1.0,
    fliplr=0.5,
    degrees=15.0,
    translate=0.1,
    scale=0.3,
)

print('\n訓練完成！')
print(f'mAP@50: {results.results_dict.get("metrics/mAP50(B)", "N/A")}')

In [ ]:
# ── Cell 8：下載 best.pt ──────────────────────────────────
import shutil
from google.colab import files

src = 'runs/train/waste_sorter/weights/best.pt'
shutil.copy(src, 'best.pt')
print(f'Model size: {os.path.getsize("best.pt") / 1e6:.1f} MB')

files.download('best.pt')
print('✅ best.pt 下載完成，接著用 SCP 傳到 Jetson：')
print('scp best.pt jetson@<JETSON_IP>:~/AI-course/models/')